In [ ]:
import json
import torch
from sentence_transformers import SentenceTransformer, util
import nltk
from nltk.corpus import stopwords

# Paths
MITRE_DICT_PATH = '/Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/MitreDictionary.json'
BING_DATA_PATH = '/Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/BingSearchData.json'

# Load MITRE ATT&CK techniques
with open(MITRE_DICT_PATH, 'r') as file:
    mitre_data = json.load(file)

# Load Bing Search Data
with open(BING_DATA_PATH, 'r') as file:
    bing_data = json.load(file)

# Precompiled stop words
nltk.download('stopwords')
STOP_WORDS = set(stopwords.words('english'))

# Select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# Build searchable corpus
technique_id_to_text = {}
for tech_id, details in mitre_data.items():
    text = (
        details.get('name', '') + ' ' +
        details.get('description', '') + ' ' +
        (' '.join(details.get('keywords', [])) + ' ') * 5
    ).lower()
    technique_id_to_text[tech_id] = text

# Precompute technique embeddings (🔥 only once!)
technique_texts = list(technique_id_to_text.values())
technique_ids = list(technique_id_to_text.keys())

print("Encoding MITRE corpus...")
technique_embeddings = model.encode(technique_texts, convert_to_tensor=True, normalize_embeddings=True)

def preprocess_text(text):
    words = text.lower().split()
    filtered_words = [word.strip('.,!?') for word in words if word not in STOP_WORDS and len(word) > 2]
    return ' '.join(filtered_words)

def find_matching_techniques_semantic(input_text, top_n=5, similarity_threshold=0.65):
    """
    Semantic matching using cosine similarity of embeddings.
    """
    processed_input = preprocess_text(input_text)
    input_embedding = model.encode(processed_input, convert_to_tensor=True, normalize_embeddings=True)

    cosine_scores = util.cos_sim(input_embedding, technique_embeddings).squeeze()

    # Find top matches
    top_scores, top_indices = torch.topk(cosine_scores, k=top_n)

    matches = []
    for idx, score in zip(top_indices, top_scores):
        if score >= similarity_threshold:
            match_id = technique_ids[idx]
            matches.append((match_id, float(score)))

    return matches

def extract_unique_tactics(matches):
    """
    Extract tactics from matched techniques.
    """
    tactics_info = []
    for match_id, score in matches:
        technique_info = mitre_data.get(match_id, {})
        technique_name = technique_info.get('name', 'Unknown')
        tactic_list = technique_info.get('tactics', [])
        for tactic in tactic_list:
            tactics_info.append((tactic, match_id, technique_name, score))
    return sorted(tactics_info)

if __name__ == "__main__":
    if not bing_data:
        print("No articles found in Bing search data.")
    else:
        first_article = bing_data[1]  # or 0 or any
        input_text = first_article.get('content', '')
        keywords = first_article.get('keywords', [])

        if not input_text:
            print("No content found in the first article.")
        else:
            combined_input = input_text + ' ' + ' '.join(keywords) * 3  # Boost keywords

            matches = find_matching_techniques_semantic(combined_input, top_n=8, similarity_threshold=0.65)
            tactics_info = extract_unique_tactics(matches)

            print(f"\nCommon MITRE ATT&CK Tactics for Article: {first_article.get('title', 'Unknown Title')}")
            for tactic, match_id, technique_name, score in tactics_info:
                print(f"- Tactic: {tactic} | Technique: {technique_name} (ID: {match_id}) | Similarity: {score:.2f}")
